# 0903 4일차

## 0. 파이썬 문법: 함수 정의

```python
def 함수명(파라미터):
    return 리턴값
```

| 요소 | 설명 |
|---|---|
| `def` | 함수를 정의하겠다는 선언 |
| `:` | **콜론 필수** - 빠뜨리면 `SyntaxError` |
| 들여쓰기 | 콜론 다음 줄부터 4칸. 들여쓰기가 곧 함수의 범위 |
| `return` | 결과를 돌려주고 함수를 끝냄 (없으면 `None`) |

→ 2일차 §0의 이름 규칙대로 함수명은 소문자가 원칙이지만, `RMSE`처럼 **약어로 굳어진 지표명**은 대문자로 쓰기도 함

### 함수 - 만드는 이유

같은 계산을 여러 번 쓸 때 식을 매번 적으면 한 군데만 고쳐도 나머지가 어긋남

```python
def RMSE(y_test, y_predict):
    return np.sqrt(mean_squared_error(y_test, y_predict))

rmse = RMSE(y_test, y_predict)
```

→ 오늘 실습 네 파일이 전부 이 함수를 그대로 재사용함

## 1. 회귀와 분류

| | 예측하는 것 | 예시 | 출력층 |
|---|---|---|---|
| **회귀** (regression) | **연속적인 수치** | 집값, 대여량, 기온 | `Dense(1)` |
| **분류** (classification) | 정해진 **라벨** 중 하나 | 개/고양이, 스팸 여부 | 라벨 개수만큼 |

→ 지금까지 실습한 것은 **전부 회귀**. 출력층이 `Dense(1)`인 이유가 "숫자 하나를 맞히는 문제"라서

→ 분류 모델은 **배우지 않은 라벨을 절대 출력할 수 없음**

### 지표 - 회귀와 분류가 갈리는 곳

분류에도 loss는 있지만 loss만으로는 "몇 개나 맞혔는지"를 알 수 없음

| | 주 지표 (loss) | 보조 지표 |
|---|---|---|
| **회귀** | `mse`, `mae` | **R²** (§3) |
| **분류** | `binary_crossentropy` 등 | **정확도(Accuracy)** |

### 용어 정리 - 전부 같은 말

> **로스(loss) = 에러(error) = 오차 = 코스트(cost) = 원값과 예측값의 차이**

## 2. 회귀의 손실 함수 - MSE, MAE, RMSE

$e = y - \hat{y}$ (실제값 - 예측값) 이라 할 때

| 지표 | 식 | 단위 |
|---|---|---|
| **MSE** | $\frac{1}{n}\sum e^2$ | y 단위의 **제곱** |
| **MAE** | $\frac{1}{n}\sum \lvert e \rvert$ | y와 **같음** |
| **RMSE** | $\sqrt{\frac{1}{n}\sum e^2}$ | y와 **같음** |

→ 둘 다 **부호 상쇄를 막는 것**이 목적 (`+3`과 `-3`을 더하면 0이 되어 "오차 없음"처럼 보임)

→ MSE는 **제곱**으로, MAE는 **절댓값**으로 음수를 없앰

### MSE - 쓰는 이유

**경사 하강법이 학습하기 좋은 모양**이기 때문 (3일차 cf)

**① 기울기가 오차 크기에 비례함**

| | 손실 | 기울기 $\partial L / \partial \hat{y}$ |
|---|---|---|
| MSE | $e^2$ | $-2e$ → **오차에 비례** |
| MAE | $\lvert e \rvert$ | $\mp 1$ → **항상 일정** |

- **MAE**: 오차가 100이든 0.01이든 보폭이 같음 → 최적점 주변에서 계속 진동
- **MSE**: 오차가 작아지면 보폭도 **자동으로 작아짐**

**② MAE는 0에서 미분이 안 됨**

$\lvert e \rvert$ 는 $e = 0$ 에서 꺾여 있어 미분값이 정의되지 않음 → 예측이 정답과 일치하는 지점에서 기울기가 끊김

**③ 크게 틀린 것부터 고침**

```
오차 [1, 1, 1, 10]
MSE 관점 -> 1, 1, 1, 100   # 10짜리를 최우선으로
MAE 관점 -> 1, 1, 1, 10    # 그냥 10배
```

**④ 통계적 근거**

오차가 정규분포를 따르면 MSE 최소화 = 최대우도추정(MLE). MSE는 **조건부 평균**, MAE는 **조건부 중앙값**을 학습

### MSE의 대가 - 이상치에 취약

위 ③의 장점이 그대로 단점이 됨. 극단값 하나가 제곱되어 loss의 대부분을 차지함

| 손실 | 언제 쓰나 |
|---|---|
| `mse` | **기본값**. 안정적으로 수렴 |
| `mae` | 이상치가 많아 신뢰할 수 없을 때 |
| `huber` | 절충안 - 작은 오차는 MSE, 큰 오차는 MAE처럼 |

### RMSE - 단위를 되돌리는 지표

MSE는 **단위가 제곱이라 값을 그대로 읽을 수 없음** (3일차 §5)

보스턴 집값은 1,000달러 단위인데 MSE의 단위는 "달러²"임

| 지표 | 값 | 의미 |
|---|---|---|
| MSE | 24.47 | 달러² - 해석 불가 |
| **RMSE** | **4.95** | **평균 약 4,950달러 빗나감** |

#### 주의) 루트를 씌워도 이상치 취약성은 남는다

RMSE는 **단위 문제만** 해결. 제곱해 평균 낸 **다음에** 루트를 씌우므로 큰 오차의 비중은 그대로

```
오차 [1, 1, 1, 10]
RMSE = sqrt((1 + 1 + 1 + 100) / 4) = 5.07   # 10 하나가 값을 지배
MAE  = (1 + 1 + 1 + 10) / 4        = 3.25   # 훨씬 덜 흔들림
```

→ 이상치 영향을 **줄이려면** RMSE가 아니라 **MAE**를 봐야 함

#### 케라스에는 RMSE가 내장돼 있지 않음

```python
model.compile(loss="mse", optimizer="adam")   # 훈련은 mse로

def RMSE(y_test, y_predict):                  # 평가 단계에서 직접 계산
    return np.sqrt(mean_squared_error(y_test, y_predict))
```

→ RMSE를 loss로 써도 최소 지점은 MSE와 같음(단조 증가 함수). 매 스텝 루트를 계산할 이유가 없어 훈련엔 `mse`를 씀

In [ ]:
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error

y_true = np.array([10, 10, 10, 10])
y_pred = np.array([11, 11, 11, 20])   # 앞 3개는 1씩, 마지막만 10 빗나감 (이상치)

mse = mean_squared_error(y_true, y_pred)
print("MSE :", mse)                          # 25.75  <- 단위가 제곱이라 해석 불가
print("RMSE:", np.sqrt(mse))                 # 5.07   <- y와 같은 단위로 복귀
print("MAE :", mean_absolute_error(y_true, y_pred))   # 3.25   <- 이상치에 덜 흔들림

# 숫자가 작아졌다고 "이상치 문제가 풀렸다"고 보면 안 됨

## 3. R² (결정계수) - 회귀의 보조지표

$$R^2 = 1 - \frac{\sum (y - \hat{y})^2}{\sum (y - \bar{y})^2}$$

| 부분 | 뜻 |
|---|---|
| 분자 | **내 모델**이 낸 오차 제곱합 |
| 분모 | **그냥 평균값으로 찍었을 때**의 오차 제곱합 |

→ 재료는 MSE와 같음. 차이는 **분모로 나눠 스케일을 없앴다**는 것

→ 그래서 R²는 단위가 없고 **데이터셋이 달라도 비교됨**

### R² 값 읽는 법

| R² | 의미 |
|---|---|
| **1** | 완벽 (오차 0) |
| 0.8 | 데이터의 흩어짐 중 80%를 모델이 설명 |
| **0** | **평균값으로 찍은 것과 동점** = 쓸모 없음 |
| **음수** | 평균보다도 못함 |

→ **하한은 없음**. 0은 최솟값이 아니라 **기준선(baseline)**

### loss가 주 지표, R²는 보조 지표

3일차 §5에서 본 문제 - **loss는 절대적 기준이 없음**

| 데이터셋 | loss (mse) | 잘한 건가? |
|---|---|---|
| 캘리포니아 | 약 0.57 | ? |
| 보스턴 | 약 22.9 | ? |
| 당뇨 | 약 2,384 | ? |

→ y 스케일이 다르니 숫자만 봐선 판단 불가. 이때 R²를 함께 보면 **같은 잣대**로 볼 수 있음

### R²가 지나치게 높으면 의심할 것

`0.999...` 같은 값이 나오면 확인:

- **과적합** - train에만 맞춰진 모델 (2일차 §4)
- **데이터 누수(leakage)** - 정답이 특성에 섞여 들어감
- **너무 쉬운 데이터** - `y = x` 같은 인위적 데이터

```python
from sklearn.metrics import r2_score

y_predict = model.predict(x_test)      # 예측을 먼저 만들어야 함
r2 = r2_score(y_test, y_predict)       # (정답, 예측값) 순서
```

→ 인자 순서 주의. 바꿔 넣어도 에러가 안 나고 값만 달라짐

## 4. evaluate와 predict

```
model.evaluate(x_test, y_test)
   └─ 내부에서 model.predict(x_test) 실행
   └─ 그 예측값과 y_test로 compile 때 지정한 loss를 계산
```

→ 방향에 주의. `predict` 안에 `evaluate`가 있는 게 **아님**

→ `model.predict()`는 **예측값만** 돌려주고 채점하지 않음

### MSE를 구하는 두 가지 경로

```python
# 경로 1 - evaluate 한 줄로 끝
loss = model.evaluate(x_test, y_test)          # loss="mse"이므로 이 값이 곧 MSE

# 경로 2 - predict 먼저, 계산은 따로
y_predict = model.predict(x_test)
mse = mean_squared_error(y_test, y_predict)
```

→ `compile(loss="mse")`로 훈련했다면 **두 값은 사실상 같음** (부동소수점 오차 수준)

### 두 경로 - 값이 달라지는 경우

| 상황 | 결과 |
|---|---|
| `compile(loss="mae")` 등 다른 손실 | `evaluate`는 MAE, sklearn은 MSE → **완전히 다른 값** |
| `metrics=['mae']` 추가 | `evaluate`가 **리스트** `[loss, mae]`를 반환 |
| y를 스케일링해서 훈련 | `evaluate`는 스케일된 공간의 loss |

### 둘 다 쓰는 이유 - RMSE와 R²

RMSE와 R²는 `evaluate`가 줄 수 없고, **둘 다 `y_predict`가 있어야** 계산됨

```python
y_predict = model.predict(x_test)               # 한 번 뽑아두고

r2   = r2_score(y_test, y_predict)
mse  = mean_squared_error(y_test, y_predict)
rmse = np.sqrt(mse)
```

→ 나란히 찍어보면 검산도 됨 (`loss="mse"`인데 값이 다르면 어딘가 잘못된 것)

In [ ]:
import numpy as np
from sklearn.metrics import mean_squared_error, r2_score

y_test    = np.array([10.0, 20.0, 30.0, 40.0])
y_predict = np.array([12.0, 18.0, 33.0, 39.0])

mse = mean_squared_error(y_test, y_predict)
print("mse  :", mse)                            # 4.5
print("rmse :", np.sqrt(mse))                   # 2.121  <- y와 같은 단위
print("r2   :", r2_score(y_test, y_predict))    # 0.964

# 평균으로만 찍었을 때 -> R²는 정확히 0
print("baseline r2 :", r2_score(y_test, np.full_like(y_test, y_test.mean())))   # 0.0

## 5. CSV와 pandas 기초

실무·대회 데이터는 대부분 **CSV 파일**로 옴

### CSV를 뷰어에서 직접 고치지 말 것

- 엑셀로 열어 손으로 수정하면 **원본이 바뀜**
- 무엇을 언제 바꿨는지 남지 않아 **재현 불가**
- 엑셀이 형식을 바꿔버리기도 함 (긴 숫자를 지수표기로, `01`을 `1`로)

→ **반드시 코드로 불러와 처리할 것**. 그래야 과정이 코드에 기록으로 남음

### pandas

```python
import pandas as pd

train_csv = pd.read_csv(path + "train.csv", index_col=0)
```

| | 설명 |
|---|---|
| `pandas` | numpy 기반의 표(테이블) 처리 라이브러리 |
| `pd.read_csv()` | **첫 줄을 자동으로 컬럼명(헤더)으로 인식**해 데이터에서 제외 |
| `index_col=0` | **0번째 컬럼을 인덱스로 지정** → 데이터에서 제외 |

#### `index_col=0`이 필요한 이유

따릉이 `train.csv`의 첫 컬럼 `id`는 순번이지 예측에 쓸 정보가 아님

```python
pd.read_csv(path + "train.csv")               # (1459, 11) - id가 특성으로 섞임
pd.read_csv(path + "train.csv", index_col=0)  # (1459, 10) - id는 인덱스로 빠짐
```

→ 안 빼면 `id`를 특성으로 착각해 학습함 → `input_dim` 숫자도 어긋남

## 6. 이상치와 결측치

### 이상치 (outlier)

전체 분포에서 비정상적으로 튀는 값. 처리는 **제거 / 값 조정 / 수집 단계에서 배제**

**절대적 기준이 아니라 상대적 기준**

→ 연봉 100억은 보통 이상치지만 **재벌 100인의 연봉 데이터**에서는 정상 범위

→ 숫자만 보고 자르지 말고 **맥락으로 판단**

### 결측치 (missing value)

값이 **아예 비어 있는** 경우. pandas에서는 `NaN`. 그대로 두면 실행 시 에러

| 처리 방법 | 코드 | 적합한 상황 |
|---|---|---|
| **삭제** | `dropna()` | 데이터가 많고 결측 비율이 작을 때 |
| **평균/중간값 채우기** | `fillna(df.mean())` | 연속적으로 변하는 값 (예: 기온) |
| **앞/뒤 값 채우기** | `ffill()` / `bfill()` | 시계열 데이터 |

```python
print(train_csv.info())          # Non-Null Count 확인
print(train_csv.isnull().sum())  # 컬럼별 결측 개수
```

#### cf) `dropna()`는 행 단위로 지운다

**한 행에 결측치가 하나라도 있으면 그 행 전체가 삭제됨**

따릉이 `train.csv`의 실제 수치

| 컬럼 | 결측 개수 |
|---|---|
| `hour_bef_pm2.5` | 117 |
| `hour_bef_pm10` | 90 |
| `hour_bef_ozone` | 76 |
| 나머지 | 2 ~ 9 |
| **컬럼별 합계** | **300** |

→ 실제로 삭제된 행은 **131개** (1459 → 1328). 결측치가 같은 행에 몰려 있었기 때문

→ 흩어져 있었다면 300행이 삭제됐을 것. **컬럼이 많을수록 많이 사라질 수 있음**

### test.csv에는 `dropna()` 금지

```python
test_csv = pd.read_csv(path + "test.csv", index_col=0)   # (715, 9)
test_csv.dropna()                                        # (674, 9)  <- 41행 삭제
```

→ `submission.csv`는 **715행**을 요구하는데 674개만 예측하면 **양식과 개수가 안 맞음**

| | 결측치 처리 | 이유 |
|---|---|---|
| **train.csv** | `dropna()` 가능 | 행이 줄어도 학습에 지장 없음 |
| **test.csv** | **`fillna()`만** | 행 개수가 제출 양식에 고정됨 |

## 7. 대회 데이터의 구조 - 따릉이

[DACON 따릉이 대여량 예측](https://dacon.io/competitions/open/235576/overview/description)

| | 데이터를 받는 방식 |
|---|---|
| **보스턴 (3일차)** | `(x_train, y_train), (x_test, y_test)` - **이미 나뉘어서** 나옴 |
| **따릉이** | CSV 3개 - **x/y 분리도, train/test 분리도 직접** |

| 파일 | 모양 | 정답(`count`) | 역할 |
|---|---|---|---|
| `train.csv` | (1459, 10) | **있음** | 학습용 |
| `test.csv` | (715, 9) | **없음** | 제출할 예측 대상 |
| `submission.csv` | (715, 1) | 전부 `NaN` | 답안지 양식 |

### train.csv를 두 번 쪼갬

```
train.csv (1459행, 정답 있음)
    ├─ ① dropna()                              1459 -> 1328행
    ├─ ② x = train_csv.drop(['count'], axis=1)  (1328, 9)  특성만
    │    y = train_csv['count']                 (1328,)    정답만
    └─ ③ train_test_split
             x_train / y_train   (훈련용 80%)
             x_test  / y_test    (채점용 20%)
```

| 코드 | 하는 일 |
|---|---|
| `drop(['count'], axis=1)` | `count` **컬럼을 제거**한 나머지 → x. `axis=1`이 "열 방향" |
| `train_csv['count']` | `count` 컬럼**만** 선택 → y. **벡터**로 나옴 `(1328,)` |

### `x_test` ≠ `test_csv` - 가장 헷갈리는 지점

| | 출처 | 정답 | 용도 |
|---|---|---|---|
| `x_test` | `train.csv`에서 떼어낸 20% | `y_test`가 **있음** | **내 모델 실력 확인** |
| `test_csv` | `test.csv` 파일 그 자체 | **없음** | **대회 제출용 예측** |

→ 정답이 있는 데이터에서 **20%를 채점용으로 떼어두는 것**

→ 정답을 알고 있으니 R², RMSE로 **채점이 가능**해짐

## 8. 전체 워크플로우

```
1. 데이터 받기        pd.read_csv(path + "train.csv", index_col=0)
2. 결측치 확인/처리    .info() -> dropna() / fillna()
3. x / y 분리         x = drop(['count'], axis=1)  |  y = train_csv['count']
4. train / test 분리   train_test_split(x, y, train_size=0.8)
5. 모델링             Sequential -> compile(loss='mse') -> fit()
6. 평가               predict(x_test) -> R², RMSE
7. 제출               test_csv 결측치 채우기 -> predict(test_csv) -> submission 저장
```

### 순서 주의 - 결측치 처리가 분리보다 먼저

| 순서 | 왜 |
|---|---|
| ① 결측치 처리 | 행을 지우는 작업이므로 **가장 먼저** |
| ② x / y 분리 | 나눈 뒤에 행을 지우면 x와 y의 행이 어긋남 |
| ③ train / test 분리 | 마지막 |

→ x와 y를 따로 `dropna()` 하면 **삭제되는 행이 달라져 짝이 깨짐**

→ `train_csv` 상태에서 한 번에 처리하고 나서 쪼개야 안전함